In [1]:
%matplotlib tk


In [5]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.widgets import Button

# =========================================================
# Fiziksel parametreler (PDF / BLACK-BOX - değiştirme)
# =========================================================
yercekimi = 9.81
dt = 0.02
top_yaricap = 0.05
cubuk_yarim_uzunluk = 1.0

# =========================================================
# Sistem durum değişkenleri (PDF ile aynı isimler)
# =========================================================
top_konum = 0.2
top_hiz = 0.0
cubuk_aci_gercek = 0.0
cubuk_aci_dinamik = 0.0
cubuk_aci_gorsel = 0.0
zaman = 0.0

# =========================================================
# Gürültü ve bozucular (PDF ile aynı)
# =========================================================
olcum_gurultusu_acik = False
fiziksel_gurultu_acik = False
cubuk_darbesi = 0.0

# =========================================================
# Kontrol modu (Sezgisel / Otomatik)
# =========================================================
AKTIF_KONTROLOR = "SEZGI"  # "SEZGI" veya "OTOMATIK"

# =========================================================
# Yardımcı fonksiyonlar
# =========================================================
def clip(x, lo, hi):
    return max(lo, min(hi, x))

def trimf(x, a, b, c):
    """Üçgensel üyelik fonksiyonu."""
    if x <= a or x >= c:
        return 0.0
    if x == b:
        return 1.0
    if x < b:
        return (x - a) / (b - a)
    return (c - x) / (c - b)

# =========================================================
# 1) SEZGİSEL BULANIK KONTROLÖR (tamamen sezgisel)
# Girdi: (top_konum, top_hiz)  Çıktı: istenen çubuk açısı
# =========================================================
def sezgisel_bulanik(top_konum_in, top_hiz_in):
    # Normalize et (konum ~[-1,1], hız için ölçek)
    x = clip(top_konum_in / cubuk_yarim_uzunluk, -1.0, 1.0)
    v = clip(top_hiz_in / 3.0, -1.0, 1.0)

    # Konum üyelikleri
    xN = trimf(x, -1.0, -1.0, 0.0)   # Sol
    xZ = trimf(x, -1.0,  0.0, 1.0)   # Orta
    xP = trimf(x,  0.0,  1.0, 1.0)   # Sağ

    # Hız üyelikleri
    vN = trimf(v, -1.0, -1.0, 0.0)   # Sola gidiyor
    vZ = trimf(v, -1.0,  0.0, 1.0)   # Yavaş
    vP = trimf(v,  0.0,  1.0, 1.0)   # Sağa gidiyor

    # Sugeno 0. derece çıkış merkezleri (sezgisel)
    NB, NS, ZO, PS, PB = -0.55, -0.25, 0.0, 0.25, 0.55

    # 3x3 kural tabanı
    rules = [
        (min(xN, vN), PB), (min(xN, vZ), PB), (min(xN, vP), PS),
        (min(xZ, vN), PS), (min(xZ, vZ), ZO), (min(xZ, vP), NS),
        (min(xP, vN), NS), (min(xP, vZ), NB), (min(xP, vP), NB),
    ]

    s = sum(w for w, _ in rules)
    u = 0.0 if s < 1e-9 else sum(w * out for w, out in rules) / s

    # Açı sınırı (mantıklı aralık)
    return clip(u, -0.9, 0.9)

# =========================================================
# 2) OTOMATİK (Random Search ile ayarlanan) BULANIK KONTROLÖR
# Kural yapısı aynı, parametreler otomatik ayarlanır.
# =========================================================
OTOMATIK_PARAMS = {
    "v_scale": 1.7,
    "NB": -0.6,
    "NS": -0.4,
    "ZO":  0.0,
    "PS":  0.4,
    "PB":  0.6,
    "u_clip": 0.9,
    # Uç bölgelerde koruma (kenar-hit'i düşürür)
    "emg_x": 0.82,
    "emg_kx": 1.4,
    "emg_kv": 0.57,
    "emg_clip": 0.75,
}

def otomatik_bulanik(top_konum_in, top_hiz_in, p=OTOMATIK_PARAMS):
    x = clip(top_konum_in / cubuk_yarim_uzunluk, -1.0, 1.0)
    v = clip(top_hiz_in / p["v_scale"], -1.0, 1.0)

    # Acil durum: top uç bölgede ve dışarı doğru gidiyorsa sert geri çevir
    if abs(x) > p["emg_x"] and (x * v) > 0:
        u_emg = -p["emg_kx"] * x - p["emg_kv"] * v
        return clip(u_emg, -p["emg_clip"], p["emg_clip"])

    xN = trimf(x, -1.0, -1.0, 0.0)
    xZ = trimf(x, -1.0,  0.0, 1.0)
    xP = trimf(x,  0.0,  1.0, 1.0)

    vN = trimf(v, -1.0, -1.0, 0.0)
    vZ = trimf(v, -1.0,  0.0, 1.0)
    vP = trimf(v,  0.0,  1.0, 1.0)

    rules = [
        (min(xN, vN), p["PB"]), (min(xN, vZ), p["PB"]), (min(xN, vP), p["PS"]),
        (min(xZ, vN), p["PS"]), (min(xZ, vZ), p["ZO"]), (min(xZ, vP), p["NS"]),
        (min(xP, vN), p["NS"]), (min(xP, vZ), p["NB"]), (min(xP, vP), p["NB"]),
    ]

    s = sum(w for w, _ in rules)
    u = 0.0 if s < 1e-9 else sum(w * out for w, out in rules) / s
    return clip(u, -p["u_clip"], p["u_clip"])

# =========================================================
# PDF'deki kontrolor fonksiyonu (SENİN DEĞİŞTİRMEN BEKLENEN BLOK)
# =========================================================
def kontrolor(top_konum_in, top_hiz_in):
    if AKTIF_KONTROLOR == "OTOMATIK":
        return otomatik_bulanik(top_konum_in, top_hiz_in)
    return sezgisel_bulanik(top_konum_in, top_hiz_in)

# =========================================================
# Simülasyon adımı (PDF - değiştirme)
# =========================================================
def adim():
    global top_konum, top_hiz
    global cubuk_aci_gercek, cubuk_aci_dinamik, cubuk_aci_gorsel
    global zaman, cubuk_darbesi

    # -------- Ölçüm --------
    olculen_konum = top_konum
    olculen_hiz = top_hiz
    if olcum_gurultusu_acik:
        olculen_konum += np.random.normal(0, 0.03)
        olculen_hiz += np.random.normal(0, 0.1)

    # -------- Kontrolör --------
    istenen_aci = kontrolor(olculen_konum, olculen_hiz)

    # -------- Aktüatör (motor) dinamiği --------
    zaman_sabiti = 0.1
    cubuk_aci_gercek += (istenen_aci - cubuk_aci_gercek) * dt / zaman_sabiti

    # -------- Fiziksel bozucu --------
    fiziksel_gurultu = 0.0
    if fiziksel_gurultu_acik:
        fiziksel_gurultu = (
            0.02 * np.sin(0.001 * zaman) +
            0.01 * np.random.randn()
        )

    # Çubuğa uygulanan ani darbenin sönümlenmesi
    cubuk_darbesi *= 0.95

    # -------- Çubuk açısının güncellenmesi --------
    cubuk_aci_dinamik = cubuk_aci_gorsel
    cubuk_aci_gorsel = cubuk_aci_gercek + fiziksel_gurultu + cubuk_darbesi

    # -------- Top dinamiği --------
    top_ivme = (5 / 7) * yercekimi * np.sin(cubuk_aci_dinamik)
    top_hiz += top_ivme * dt
    top_konum += top_hiz * dt

    # Topun çubuktan düşmesini engelle
    top_konum = np.clip(top_konum, -cubuk_yarim_uzunluk, cubuk_yarim_uzunluk)

    zaman += dt

# =========================================================
# Performans metrikleri (ödev gereği)
# =========================================================
def kosu_olc(toplam_sure=8.0, senaryo="S1"):
    """
    Animasyonsuz koşu:
    - RMSE (merkezlenme başarısı)
    - max|x| (aşım)
    - kenar-hit (uçlara vurma / çok yaklaşma sayısı)
    """
    global top_konum, top_hiz, cubuk_aci_gercek, cubuk_aci_dinamik, cubuk_aci_gorsel, zaman, cubuk_darbesi
    global olcum_gurultusu_acik, fiziksel_gurultu_acik

    # yedek
    backup = (top_konum, top_hiz, cubuk_aci_gercek, cubuk_aci_dinamik, cubuk_aci_gorsel, zaman, cubuk_darbesi,
              olcum_gurultusu_acik, fiziksel_gurultu_acik)

    # reset
    top_konum = 0.2
    top_hiz = 0.0
    cubuk_aci_gercek = 0.0
    cubuk_aci_dinamik = 0.0
    cubuk_aci_gorsel = 0.0
    cubuk_darbesi = 0.0
    zaman = 0.0

    # senaryo
    if senaryo == "S1":
        olcum_gurultusu_acik = False
        fiziksel_gurultu_acik = False
    else:
        olcum_gurultusu_acik = True
        fiziksel_gurultu_acik = True

    darbe_an = 1.0
    darbe_deger = 1.0

    xs = []
    edge_hits = 0

    n = int(toplam_sure / dt)
    for k in range(n):
        t = k * dt
        if abs(t - darbe_an) < 1e-12:
            cubuk_darbesi = darbe_deger

        adim()
        xs.append(top_konum)
        if abs(top_konum) > 0.99 * cubuk_yarim_uzunluk:
            edge_hits += 1

    xs = np.array(xs)
    rmse = float(np.sqrt(np.mean(xs**2)))
    maxabs = float(np.max(np.abs(xs)))

    # geri yükle
    (top_konum, top_hiz, cubuk_aci_gercek, cubuk_aci_dinamik, cubuk_aci_gorsel, zaman, cubuk_darbesi,
     olcum_gurultusu_acik, fiziksel_gurultu_acik) = backup

    return rmse, maxabs, edge_hits

def random_search_otomatik(iterasyon=150, seed=0):
    """
    Otomatik kontrolör parametrelerini random search ile ayarla.
    Amaç: (RMSE + 0.5*max|x| + 0.02*kenar-hit) minimize.
    """
    global OTOMATIK_PARAMS, AKTIF_KONTROLOR
    rng = np.random.default_rng(seed)

    best_score = 1e18
    best = None

    old_mode = AKTIF_KONTROLOR
    AKTIF_KONTROLOR = "OTOMATIK"
    old_params = OTOMATIK_PARAMS

    for _ in range(iterasyon):
        cand = {
            "v_scale": float(rng.uniform(1.0, 2.5)),
            "NB": float(-rng.uniform(0.45, 0.75)),
            "NS": float(-rng.uniform(0.25, 0.55)),
            "ZO": 0.0,
            "PS": float(rng.uniform(0.25, 0.55)),
            "PB": float(rng.uniform(0.45, 0.75)),
            "u_clip": float(rng.uniform(0.7, 1.1)),
            "emg_x": float(rng.uniform(0.70, 0.90)),
            "emg_kx": float(rng.uniform(0.8, 2.0)),
            "emg_kv": float(rng.uniform(0.2, 1.2)),
            "emg_clip": float(rng.uniform(0.5, 0.95)),
        }
        OTOMATIK_PARAMS = cand

        r1, m1, e1 = kosu_olc(senaryo="S1")
        r2, m2, e2 = kosu_olc(senaryo="S2")
        score = (r1 + r2) + 0.5 * (m1 + m2) + 0.02 * (e1 + e2)

        if score < best_score:
            best_score = score
            best = cand

    # en iyiyi set et
    if best is not None:
        OTOMATIK_PARAMS = best
    else:
        OTOMATIK_PARAMS = old_params

    AKTIF_KONTROLOR = old_mode
    return best_score, best

def karsilastirma_yazdir():
    global AKTIF_KONTROLOR
    print("Otomatik kontrolör ayarlanıyor (random search)...")
    score, best = random_search_otomatik(iterasyon=150, seed=0)
    print(f"Otomatik kontrolör bulundu. Skor: {score:.4f}")
    print("OTOMATIK_PARAMS =", best)

    print("\n=== KARŞILAŞTIRMA (aynı senaryolar) ===")
    for senaryo in ["S1", "S2"]:
        print(f"\n{senaryo}: {'Gürültüsüz + Darbe' if senaryo=='S1' else 'Ölçüm + Fiziksel Gürültü + Darbe'}")

        AKTIF_KONTROLOR = "SEZGI"
        r, m, e = kosu_olc(senaryo=senaryo)
        print(f"  Sezgisel -> RMSE: {r:.4f}, max|x|: {m:.3f}, kenar-hit: {e}")

        AKTIF_KONTROLOR = "OTOMATIK"
        r, m, e = kosu_olc(senaryo=senaryo)
        print(f"  Otomatik -> RMSE: {r:.4f}, max|x|: {m:.3f}, kenar-hit: {e}")

    # animasyona sezgisel başla
    AKTIF_KONTROLOR = "SEZGI"

# =========================================================
# Görselleştirme (PDF + kontrolör bilgisi eklendi)
# =========================================================
fig, ax = plt.subplots()
plt.subplots_adjust(bottom=0.36)
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-0.6, 0.6)
ax.set_aspect('equal')

cubuk_cizgi, = ax.plot([], [], lw=4)
top_nokta, = ax.plot([], [], 'o', markersize=14)
bilgi_yazisi = ax.text(-1.15, 0.48, "", fontsize=9)

def guncelle(frame):
    adim()

    # Çubuk çizimi
    x1 = -cubuk_yarim_uzunluk * np.cos(cubuk_aci_gorsel)
    y1 = -cubuk_yarim_uzunluk * np.sin(cubuk_aci_gorsel)
    x2 =  cubuk_yarim_uzunluk * np.cos(cubuk_aci_gorsel)
    y2 =  cubuk_yarim_uzunluk * np.sin(cubuk_aci_gorsel)
    cubuk_cizgi.set_data([x1, x2], [y1, y2])

    # Top çizimi
    top_x = top_konum * np.cos(cubuk_aci_gorsel)
    top_y = top_konum * np.sin(cubuk_aci_gorsel)
    normal_x = -np.sin(cubuk_aci_gorsel)
    normal_y =  np.cos(cubuk_aci_gorsel)
    top_nokta.set_data(
        top_x + top_yaricap * normal_x,
        top_y + top_yaricap * normal_y
    )

    bilgi_yazisi.set_text(
        f"Aktif kontrolör: {AKTIF_KONTROLOR}\n"
        f"Ölçüm gürültüsü: {olcum_gurultusu_acik}\n"
        f"Fiziksel gürültü: {fiziksel_gurultu_acik}\n"
        f"Konum: {top_konum:+.3f}  Hız: {top_hiz:+.3f}"
    )

    return cubuk_cizgi, top_nokta, bilgi_yazisi

# =========================================================
# Buton fonksiyonları (PDF + Sezgi/Oto eklendi)
# =========================================================
def cubuga_darbe(event):
    global cubuk_darbesi
    cubuk_darbesi = 1.0

def olcum_gurultusu_degistir(event):
    global olcum_gurultusu_acik
    olcum_gurultusu_acik = not olcum_gurultusu_acik

def fiziksel_gurultu_degistir(event):
    global fiziksel_gurultu_acik
    fiziksel_gurultu_acik = not fiziksel_gurultu_acik

def sezgi_oto_degistir(event):
    global AKTIF_KONTROLOR
    AKTIF_KONTROLOR = "OTOMATIK" if AKTIF_KONTROLOR == "SEZGI" else "SEZGI"

def sifirla(event):
    global top_konum, top_hiz
    global cubuk_aci_gercek, cubuk_aci_dinamik, cubuk_aci_gorsel
    global zaman, cubuk_darbesi
    top_konum = 0.2
    top_hiz = 0.0
    cubuk_aci_gercek = 0.0
    cubuk_aci_dinamik = 0.0
    cubuk_aci_gorsel = 0.0
    cubuk_darbesi = 0.0
    zaman = 0.0

# =========================================================
# Butonlar (PDF yerleşimi + Sezgi/Oto)
# =========================================================
ax_darbe = plt.axes([0.08, 0.18, 0.18, 0.08])
ax_olcum = plt.axes([0.30, 0.18, 0.18, 0.08])
ax_fizik = plt.axes([0.52, 0.18, 0.18, 0.08])
ax_mode  = plt.axes([0.74, 0.18, 0.18, 0.08])
ax_reset = plt.axes([0.74, 0.06, 0.18, 0.08])

btn_darbe = Button(ax_darbe, "Çubuğa Darbe")
btn_olcum = Button(ax_olcum, "Ölçüm Gürültüsü")
btn_fizik = Button(ax_fizik, "Fiziksel Gürültü")
btn_mode  = Button(ax_mode,  "Sezgi/Oto")
btn_reset = Button(ax_reset, "Sıfırla")

btn_darbe.on_clicked(cubuga_darbe)
btn_olcum.on_clicked(olcum_gurultusu_degistir)
btn_fizik.on_clicked(fiziksel_gurultu_degistir)
btn_mode.on_clicked(sezgi_oto_degistir)
btn_reset.on_clicked(sifirla)

# =========================================================
# 1) Önce metrik karşılaştırması yazdır
# 2) Sonra animasyonu başlat
# =========================================================
karsilastirma_yazdir()
animasyon = FuncAnimation(fig, guncelle, interval=20, cache_frame_data=False)
plt.show()


Otomatik kontrolör ayarlanıyor (random search)...
Otomatik kontrolör bulundu. Skor: 1.2091
OTOMATIK_PARAMS = {'v_scale': 1.3517141495869605, 'NB': -0.5021921732521362, 'NS': -0.36660430483461215, 'ZO': 0.0, 'PS': 0.4531057082265901, 'PB': 0.45443100147583504, 'u_clip': 0.7553580990704577, 'emg_x': 0.8616216242741167, 'emg_kx': 1.1985743499677193, 'emg_kv': 0.7591559142258661, 'emg_clip': 0.5249721584623346}

=== KARŞILAŞTIRMA (aynı senaryolar) ===

S1: Gürültüsüz + Darbe
  Sezgisel -> RMSE: 0.4835, max|x|: 0.935, kenar-hit: 0
  Otomatik -> RMSE: 0.2608, max|x|: 0.712, kenar-hit: 0

S2: Ölçüm + Fiziksel Gürültü + Darbe
  Sezgisel -> RMSE: 0.4792, max|x|: 0.920, kenar-hit: 0
  Otomatik -> RMSE: 0.2620, max|x|: 0.721, kenar-hit: 0
